# CTAB-GAN+ discriminator snapshot analysis

This notebook displays the internal CTAB-GAN+ discriminator snapshot diagnostics and compares their GradientSHAP attributions with the post-hoc Random Forest detector's TreeSHAP values. The primary comparison population is **correctly classified synthetic holdout rows** for both models.

A0 is the same-variant comparison. Both models use the same deterministic holdout and correct-synthetic rule, although their correctly classified row subsets can differ. A1–A5 snapshot rankings can also be compared with the common A0 detector reference used to construct the ablation priorities, but those comparisons are explicitly marked as cross-variant diagnostics. Historical discriminators are evaluated against a fixed sample from their variant's final generator, not an epoch-matched generator.

## 1. Controls

In [ ]:
RUN_DIRECTORY = ""  # Result-directory name, relative path, or absolute path; empty selects latest compatible run.
RESULTS_ROOT = None  # Optional path override; defaults to PROJECT_ROOT / 'results'.
VARIANTS = None  # Example: ['A0', 'A5']; None discovers all available variants.
COMPARE_VARIANT = "A0"
TOP_N = 12
TOP_K = 5

## 2. Locate the project and load snapshot artifacts

In [ ]:
from pathlib import Path
from IPython.display import display, Markdown
import json
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")

def find_project_root(start):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "xai_reweighting").is_dir() and (candidate / "configs").is_dir():
            return candidate
        nested = candidate / "CTAB-GAN-Plus-main"
        if (nested / "xai_reweighting").is_dir():
            return nested
    raise FileNotFoundError("Could not locate CTAB-GAN-Plus-main")

def variant_key(value):
    match = re.fullmatch(r"A(\d+)", str(value))
    return int(match.group(1)) if match else 10_000

PROJECT_ROOT = find_project_root(Path.cwd())
results_root = Path(RESULTS_ROOT).expanduser() if RESULTS_ROOT else PROJECT_ROOT / "results"
if not results_root.is_absolute():
    results_root = (PROJECT_ROOT / results_root).resolve()

def resolve_run_directory(value):
    if value:
        candidate = Path(value).expanduser()
        choices = [candidate] if candidate.is_absolute() else [results_root / candidate, PROJECT_ROOT / candidate]
        for choice in choices:
            if choice.is_dir():
                return choice.resolve()
        raise FileNotFoundError(f"Result directory not found: {value}")
    matches = [path.parent for path in results_root.rglob("discriminator_shap_A0.csv")]
    if not matches:
        raise FileNotFoundError(f"No snapshot-enabled run found below {results_root}")
    return max(matches, key=lambda path: path.stat().st_mtime).resolve()

run_directory = resolve_run_directory(RUN_DIRECTORY)
pattern = re.compile(r"discriminator_shap_(A\d+)\.csv$")
discovered = []
for path in run_directory.glob("discriminator_shap_A*.csv"):
    match = pattern.fullmatch(path.name)
    if match:
        discovered.append(match.group(1))
available_variants = sorted(set(discovered), key=variant_key)
selected_variants = available_variants if VARIANTS is None else [v for v in VARIANTS if v in available_variants]
missing_requested = [] if VARIANTS is None else [v for v in VARIANTS if v not in available_variants]
if not selected_variants:
    raise FileNotFoundError("No requested discriminator snapshot artifacts are present")

def parse_bool(series):
    return series.astype(str).str.strip().str.lower().isin({"true", "1", "yes"})

shap_parts, metric_parts, prediction_parts = [], [], []
for variant in selected_variants:
    shap_frame = pd.read_csv(run_directory / f"discriminator_shap_{variant}.csv")
    shap_frame.insert(0, "variant", variant)
    if "primary_scope" in shap_frame:
        shap_frame["primary_scope"] = parse_bool(shap_frame["primary_scope"])
    shap_parts.append(shap_frame)
    metrics_path = run_directory / f"discriminator_snapshot_metrics_{variant}.csv"
    if metrics_path.exists():
        frame = pd.read_csv(metrics_path)
        frame.insert(0, "variant", variant)
        metric_parts.append(frame)
    predictions_path = run_directory / f"discriminator_snapshot_predictions_{variant}.csv"
    if predictions_path.exists():
        frame = pd.read_csv(predictions_path)
        frame.insert(0, "variant", variant)
        prediction_parts.append(frame)

snapshot_shap = pd.concat(shap_parts, ignore_index=True)
snapshot_metrics = pd.concat(metric_parts, ignore_index=True) if metric_parts else pd.DataFrame()
snapshot_predictions = pd.concat(prediction_parts, ignore_index=True) if prediction_parts else pd.DataFrame()
detector_path = run_directory / "baseline_detector_shap.csv"
if not detector_path.exists():
    raise FileNotFoundError(f"Missing post-hoc detector SHAP artifact: {detector_path}")
detector_shap = pd.read_csv(detector_path)

display(Markdown(f"### Selected run: `{run_directory.name}`"))
display(pd.DataFrame({
    "value": [str(run_directory), ', '.join(available_variants), ', '.join(selected_variants), ', '.join(missing_requested) or 'none'],
}, index=["run directory", "available variants", "selected variants", "missing requested variants"]))

## 3. Snapshot classification performance

AUC and average precision are threshold-independent. Accuracy and balanced accuracy use the threshold selected on the separate calibration portion of the shared audit probe.

In [ ]:
if snapshot_metrics.empty:
    display(Markdown("No snapshot metric artifacts are available."))
else:
    metric_columns = [column for column in ["auc", "average_precision", "accuracy", "balanced_accuracy"] if column in snapshot_metrics]
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    axes = axes.ravel()
    for axis, metric in zip(axes, metric_columns):
        sns.lineplot(data=snapshot_metrics, x="epoch", y=metric, hue="variant", marker="o", hue_order=selected_variants, ax=axis)
        axis.set_title(metric.replace('_', ' ').title())
        axis.set_xlabel("Snapshot epoch")
        axis.set_ylabel(metric.replace('_', ' ').title())
        axis.set_ylim(0, 1.02)
    for axis in axes[len(metric_columns):]:
        axis.set_visible(False)
    fig.suptitle("Internal discriminator performance on the shared audit holdout", fontsize=15)
    fig.tight_layout()
    plt.show()
    final_metrics = snapshot_metrics.loc[snapshot_metrics.groupby("variant")["epoch"].idxmax()].sort_values("variant", key=lambda s: s.map(variant_key))
    display(Markdown("#### Final snapshot metrics"))
    display(final_metrics.round(4))

In [ ]:
if not snapshot_metrics.empty:
    outcome_columns = [column for column in snapshot_metrics if column.startswith("rows_")]
    if outcome_columns:
        outcome_long = snapshot_metrics.melt(id_vars=["variant", "epoch", "holdout_rows"], value_vars=outcome_columns, var_name="outcome_group", value_name="rows")
        outcome_long["outcome_group"] = outcome_long["outcome_group"].str.removeprefix("rows_")
        outcome_long["holdout_fraction"] = outcome_long["rows"] / outcome_long["holdout_rows"]
        final_outcomes = outcome_long.loc[outcome_long.groupby(["variant", "outcome_group"])["epoch"].idxmax()]
        pivot = final_outcomes.pivot(index="variant", columns="outcome_group", values="holdout_fraction").reindex(selected_variants)
        fig, axis = plt.subplots(figsize=(11, max(3, 0.65 * len(pivot) + 2)))
        sns.heatmap(pivot, annot=True, fmt=".1%", cmap="Blues", vmin=0, vmax=0.5, ax=axis)
        axis.set_title("Final snapshot classification-outcome fractions")
        axis.set_xlabel("Classification outcome group")
        axis.set_ylabel("Ablation variant")
        fig.tight_layout()
        plt.show()

## 4. Outcome-conditioned snapshot explanations

The heatmaps keep the four classification outcomes separate. Positive signed SHAP pushes the critic toward **real**; negative signed SHAP pushes it toward **synthetic**. Importance magnitude is computed after grouping encoded dimensions back to each original feature.

In [ ]:
analysis_variant = COMPARE_VARIANT if COMPARE_VARIANT in selected_variants else selected_variants[0]
variant_shap = snapshot_shap[snapshot_shap["variant"] == analysis_variant].copy()
final_epoch = variant_shap["epoch"].max()
final_shap = variant_shap[variant_shap["epoch"] == final_epoch].copy()
primary_final = final_shap[final_shap["primary_scope"]].sort_values("importance_share", ascending=False)
top_features = primary_final.head(TOP_N)["feature"].tolist()

magnitude = final_shap[final_shap["feature"].isin(top_features)].pivot(index="outcome_group", columns="feature", values="importance_share").reindex(columns=top_features)
signed = final_shap[final_shap["feature"].isin(top_features)].pivot(index="outcome_group", columns="feature", values="mean_signed_shap").reindex(columns=top_features)
fig, axes = plt.subplots(2, 1, figsize=(max(12, TOP_N * 0.9), 8))
sns.heatmap(magnitude, annot=True, fmt=".3f", cmap="Blues", ax=axes[0])
axes[0].set_title(f"{analysis_variant} epoch {final_epoch}: importance share by outcome group")
axes[0].set_xlabel("Original feature")
axes[0].set_ylabel("Classification outcome group")
limit = np.nanmax(np.abs(signed.to_numpy())) if signed.notna().any().any() else 1.0
limit = limit if np.isfinite(limit) and limit > 0 else 1.0
sns.heatmap(signed, annot=True, fmt=".3g", cmap="vlag", center=0, vmin=-limit, vmax=limit, ax=axes[1])
axes[1].set_title(f"{analysis_variant} epoch {final_epoch}: mean signed SHAP by outcome group")
axes[1].set_xlabel("Original feature")
axes[1].set_ylabel("Classification outcome group")
fig.tight_layout()
plt.show()

In [ ]:
primary_trajectory = variant_shap[variant_shap["primary_scope"] & variant_shap["feature"].isin(top_features)].copy()
fig, axis = plt.subplots(figsize=(13, 6))
sns.lineplot(data=primary_trajectory, x="epoch", y="importance_share", hue="feature", marker="o", hue_order=top_features, ax=axis)
axis.set_title(f"{analysis_variant}: correctly classified synthetic SHAP trajectory")
axis.set_xlabel("Snapshot epoch")
axis.set_ylabel("Normalized mean absolute GradientSHAP importance")
axis.legend(title="Original feature", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
plt.show()
display(Markdown(f"#### Final correctly classified synthetic ranking: {analysis_variant}, epoch {final_epoch}"))
display(primary_final[["rank", "feature", "importance_share", "mean_abs_shap", "mean_signed_shap", "candidate_rows", "explained_rows"]].head(TOP_N).round(5))

## 5. Snapshot SHAP versus post-hoc detector SHAP

The shared semantic scope is correctly classified synthetic rows from the same deterministic audit holdout. Each model determines correctness using its own prediction, so the exact selected row subset can differ. The post-hoc detector explains the probability of `real`; the internal critic is oriented so larger scores indicate `real`. Therefore their signed directions have the same interpretation. A0 is the same-variant comparison. Other variants use the fixed A0 detector as a reference because the ablation framework deliberately derives all weighting signals from A0.

In [ ]:
required_detector_columns = {"feature", "importance_share", "mean_signed_shap"}
missing_detector_columns = required_detector_columns - set(detector_shap.columns)
if missing_detector_columns:
    raise ValueError(f"Detector SHAP artifact is missing columns: {sorted(missing_detector_columns)}")

primary = snapshot_shap[snapshot_shap["primary_scope"]].copy()
comparison_rows = []
comparison_frames = {}
detector_reference = detector_shap.set_index("feature")
for (variant, epoch), group in primary.groupby(["variant", "epoch"], sort=False):
    snapshot_reference = group.set_index("feature")
    features = sorted(set(detector_reference.index.astype(str)) | set(snapshot_reference.index.astype(str)))
    comparison = pd.DataFrame({"feature": features})
    comparison["detector_importance_share"] = comparison["feature"].map(detector_reference["importance_share"]).fillna(0.0)
    comparison["snapshot_importance_share"] = comparison["feature"].map(snapshot_reference["importance_share"]).fillna(0.0)
    comparison["detector_mean_signed_shap"] = comparison["feature"].map(detector_reference["mean_signed_shap"])
    comparison["snapshot_mean_signed_shap"] = comparison["feature"].map(snapshot_reference["mean_signed_shap"])
    comparison["detector_rank"] = comparison["detector_importance_share"].rank(method="min", ascending=False).astype(int)
    comparison["snapshot_rank"] = comparison["snapshot_importance_share"].rank(method="min", ascending=False).astype(int)
    valid_sign = comparison[["detector_mean_signed_shap", "snapshot_mean_signed_shap"]].notna().all(axis=1)
    valid_sign &= ~np.isclose(comparison["detector_mean_signed_shap"], 0) & ~np.isclose(comparison["snapshot_mean_signed_shap"], 0)
    comparison["signed_direction_agreement"] = np.where(valid_sign, np.sign(comparison["detector_mean_signed_shap"]) == np.sign(comparison["snapshot_mean_signed_shap"]), np.nan)
    detector_top = set(comparison.nlargest(min(TOP_K, len(comparison)), "detector_importance_share")["feature"])
    snapshot_top = set(comparison.nlargest(min(TOP_K, len(comparison)), "snapshot_importance_share")["feature"])
    union = detector_top | snapshot_top
    correlation = comparison["detector_importance_share"].corr(comparison["snapshot_importance_share"], method="spearman")
    comparison_rows.append({
        "variant": variant, "epoch": epoch,
        "comparison_scope": "direct_A0" if variant == "A0" else "cross_variant_A0_detector_reference",
        "spearman_importance_correlation": correlation,
        "top_k_overlap_count": len(detector_top & snapshot_top),
        "top_k_jaccard": len(detector_top & snapshot_top) / len(union) if union else 0.0,
        "signed_direction_agreement_rate": pd.to_numeric(comparison.loc[valid_sign, "signed_direction_agreement"], errors="coerce").mean() if valid_sign.any() else np.nan,
    })
    comparison_frames[(variant, int(epoch))] = comparison
comparison_summary = pd.DataFrame(comparison_rows).sort_values(["variant", "epoch"], key=lambda s: s.map(variant_key) if s.name == "variant" else s)
display(comparison_summary.round(4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
comparison_metrics = ["spearman_importance_correlation", "top_k_jaccard", "signed_direction_agreement_rate"]
titles = ["Importance-rank correlation", f"Top-{TOP_K} Jaccard overlap", "Signed-direction agreement"]
for axis, metric, title in zip(axes, comparison_metrics, titles):
    sns.lineplot(data=comparison_summary, x="epoch", y=metric, hue="variant", marker="o", hue_order=selected_variants, ax=axis)
    axis.set_title(title)
    axis.set_xlabel("Snapshot epoch")
    axis.set_ylabel(title)
    axis.set_ylim(-1.02 if metric.startswith('spearman') else 0, 1.02)
fig.suptitle("Internal discriminator SHAP versus the A0 post-hoc detector reference", fontsize=15)
fig.tight_layout()
plt.show()

In [ ]:
compare_variant = COMPARE_VARIANT if COMPARE_VARIANT in selected_variants else selected_variants[0]
compare_epoch = int(primary.loc[primary["variant"] == compare_variant, "epoch"].max())
final_comparison = comparison_frames[(compare_variant, compare_epoch)].copy()
final_comparison["absolute_importance_gap"] = (final_comparison["snapshot_importance_share"] - final_comparison["detector_importance_share"]).abs()
label_features = set(final_comparison.nlargest(TOP_N, "absolute_importance_gap")["feature"])
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.scatterplot(data=final_comparison, x="detector_importance_share", y="snapshot_importance_share", hue="signed_direction_agreement", palette={True: "#2ca02c", False: "#d62728"}, ax=axes[0])
maximum = max(final_comparison["detector_importance_share"].max(), final_comparison["snapshot_importance_share"].max(), 1e-9)
axes[0].plot([0, maximum], [0, maximum], linestyle="--", color="black", linewidth=1)
for row in final_comparison.itertuples():
    if row.feature in label_features:
        axes[0].annotate(row.feature, (row.detector_importance_share, row.snapshot_importance_share), fontsize=8, xytext=(3, 3), textcoords="offset points")
axes[0].set_title(f"{compare_variant} epoch {compare_epoch}: importance shares")
axes[0].set_xlabel("Post-hoc detector TreeSHAP importance share")
axes[0].set_ylabel("Internal discriminator GradientSHAP importance share")
rank_plot = final_comparison.nsmallest(TOP_N, "detector_rank").sort_values("detector_rank", ascending=False)
y = np.arange(len(rank_plot))
axes[1].barh(y - 0.2, rank_plot["detector_importance_share"], height=0.4, label="Post-hoc detector")
axes[1].barh(y + 0.2, rank_plot["snapshot_importance_share"], height=0.4, label="Internal discriminator")
axes[1].set_yticks(y, rank_plot["feature"])
axes[1].set_title(f"Top {TOP_N} detector features compared")
axes[1].set_xlabel("Normalized mean absolute SHAP importance")
axes[1].set_ylabel("Original feature")
axes[1].legend()
fig.tight_layout()
plt.show()
display(Markdown(f"#### Largest ranking disagreements for {compare_variant}, epoch {compare_epoch}"))
display(final_comparison.assign(rank_gap=lambda frame: (frame["snapshot_rank"] - frame["detector_rank"]).abs()).nlargest(TOP_N, "rank_gap").round(4))

## 6. Final-snapshot feature comparison across ablation variants

This view uses the common A0 detector ranking as the column order. It shows whether weighted retraining changes which artifacts the internal discriminator relies on. It does **not** imply that a new Random Forest detector was fitted for every variant.

In [ ]:
detector_top_features = detector_shap.nlargest(TOP_N, "importance_share")["feature"].tolist()
final_primary_parts = []
for variant in selected_variants:
    subset = primary[primary["variant"] == variant]
    epoch = subset["epoch"].max()
    final_primary_parts.append(subset[subset["epoch"] == epoch])
final_primary = pd.concat(final_primary_parts, ignore_index=True)
snapshot_heatmap = final_primary[final_primary["feature"].isin(detector_top_features)].pivot(index="variant", columns="feature", values="importance_share").reindex(index=selected_variants, columns=detector_top_features)
detector_row = detector_shap.set_index("feature")["importance_share"].reindex(detector_top_features).to_frame().T
detector_row.index = ["A0 post-hoc detector"]
combined_heatmap = pd.concat([detector_row, snapshot_heatmap])
fig, axis = plt.subplots(figsize=(max(12, TOP_N * 0.9), max(4, len(combined_heatmap) * 0.65 + 2)))
sns.heatmap(combined_heatmap, annot=True, fmt=".3f", cmap="YlGnBu", ax=axis)
axis.set_title("A0 detector reference and final internal-discriminator importance shares")
axis.set_xlabel("Feature, ordered by A0 post-hoc detector importance")
axis.set_ylabel("Model / ablation variant")
fig.tight_layout()
plt.show()

## Interpretation checklist

- High AUC means the discriminator can still separate real and synthetic rows; it is not itself evidence of good generation.
- A feature important for both correct-real and correct-synthetic rows is a bidirectional distribution separator.
- A feature important mainly for correct-synthetic rows is more consistent with a synthetic artifact.
- Opposing signed SHAP values are expected when a feature pushes real rows toward real and synthetic rows toward synthetic.
- Low detector/snapshot agreement does not automatically invalidate either explanation: the Random Forest and GAN critic have different model classes and training objectives.
- Both models use the same deterministic holdout and the same correct-synthetic rule, but the rows satisfying that rule may differ between models.
- Cross-variant comparisons against the A0 detector are diagnostics of how the internal critic changed; only A0 is a same-variant detector comparison.